**To Download the Library.**

In [1]:
pip install requests beautifulsoup4 python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00


**To Scrap the Wesite Run This Function and Enter The Domain URL .**

**To Download the File Which is Been Scrapped.**

In [11]:
import requests
from bs4 import BeautifulSoup
from docx import Document
from urllib.parse import urljoin, urlparse
from docx.shared import Pt
import re
import os
from google.colab import files
from tqdm import tqdm   # 🔹 Progress bar


# ------------------ HELPERS ------------------

def clean_text_for_docx(text):
    if not text:
        return ""
    text = text.replace('\x00', '')
    illegal_xml_chars = re.compile(
        u'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f\ufffe\uffff]'
    )
    return illegal_xml_chars.sub('', text)


def normalize_domain(domain):
    return domain.lower().replace("www.", "")


# ------------------ CONTENT EXTRACTION ------------------

def add_content_to_doc(soup, doc, url):
    doc.add_heading(f"Scraped Page: {url}", level=1)

    for tag in soup(['script', 'style', 'nav', 'footer', 'noscript']):
        tag.decompose()

    main_content = soup.find('main') or soup.body
    if not main_content:
        return

    for element in main_content.find_all(['h1', 'h2', 'h3', 'p', 'li']):
        text = clean_text_for_docx(element.get_text(" ", strip=True))

        if len(text) < 10:
            continue

        if element.name in ['h1', 'h2']:
            doc.add_heading(text, level=2)
        elif element.name == 'h3':
            doc.add_heading(text, level=3)
        elif element.name == 'li':
            doc.add_paragraph(text, style='List Bullet')
        else:
            p = doc.add_paragraph(text)
            p.paragraph_format.space_after = Pt(6)


# ------------------ LINK DISCOVERY ------------------

def find_unique_links(soup, base_url, base_domain, visited):
    new_links = set()

    for a in soup.find_all('a', href=True):
        full_url = urljoin(base_url, a['href']).split('#')[0]
        parsed = urlparse(full_url)

        if normalize_domain(parsed.netloc) != base_domain:
            continue

        if full_url in visited:
            continue

        if parsed.path.lower().endswith(('.pdf', '.jpg', '.png', '.zip')):
            continue

        new_links.add(full_url)

    return new_links


# ------------------ SCRAPER CORE ------------------

def recursive_scrape(start_url):
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    base_domain = normalize_domain(urlparse(start_url).netloc)

    doc = Document()
    doc.add_heading(f"Website Scrape: {base_domain}", 0)

    to_visit = {start_url}
    visited = set()

    progress = tqdm(desc="Pages scraped", unit="page")

    while to_visit:
        url = to_visit.pop()
        if url in visited:
            continue

        visited.add(url)
        progress.update(1)
        progress.set_postfix(scraped=len(visited), queue=len(to_visit))

        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if "text/html" not in r.headers.get("Content-Type", ""):
                continue

            soup = BeautifulSoup(r.text, "html.parser")
            add_content_to_doc(soup, doc, url)
            doc.add_page_break()

            new_links = find_unique_links(soup, url, base_domain, visited)
            to_visit.update(new_links)

        except Exception as e:
            print(f"Failed: {e}")

    progress.close()
    print(f"\nTotal pages scraped: {len(visited)}")
    return doc


# ------------------ RUN ------------------

if __name__ == "__main__":

    start_url = input("Enter URL to scrape: ").strip()

    file_name = input("Enter file name to save (with or without .docx): ").strip()
    if not file_name.lower().endswith(".docx"):
        file_name += ".docx"

    document = recursive_scrape(start_url)
    document.save(file_name)

    print(f"\n✅ File saved successfully as: {file_name}")

    if os.path.exists(file_name):
        files.download(file_name)


Enter URL to scrape: https://bigbear.ai/
Enter file name to save (with or without .docx): B


Pages scraped: 493page [06:05,  2.75s/page, queue=15, scraped=493]

Failed: HTTPSConnectionPool(host='bigbear.ai', port=443): Read timed out. (read timeout=10)


Pages scraped: 509page [06:18,  1.34page/s, queue=0, scraped=509]


Total pages scraped: 509

✅ File saved successfully as: B.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Optional Download Case for a Document From Google Colab**

In [13]:
from google.colab import files
files.download('scraped_content.docx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>